# MRIxFields2026 Task 3 — training and training results

This notebook is only for future Task 3 training and its local training/holdout
results. It contains no organizer-pretrained validation sweep and no submission
builder. Training remains disabled until the organizer-pretrained baseline
milestone is complete and explicitly acknowledged.

Mounted Drive supplies source code, source data, and an optional initialization
checkpoint. Each required item is copied into /content before use. Preprocessing,
caches, logs, checkpoints, and metrics stay in runtime storage. The final cell
is the only opt-in Drive write.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

RUNTIME_ROOT_VALUE = '/content/mrixfields_task3/training'  # @param {type:'string'}
DRIVE_CODE_SOURCE_VALUE = '/content/drive/MyDrive/mrixfields_task3_source'  # @param {type:'string'}
DRIVE_PROSPECTIVE_SOURCE_VALUE = '/content/drive/MyDrive/mrixfields2026/training_prospective'  # @param {type:'string'}
DRIVE_RETROSPECTIVE_SOURCE_VALUE = '/content/drive/MyDrive/mrixfields2026/training_retrospective'  # @param {type:'string'}
DRIVE_INITIAL_CHECKPOINT_VALUE = '/content/drive/MyDrive/mrixfields_weights/task3_any_to_any_multimodal/stargan_v2/pro_pretrained/weights/checkpoint_epoch50.pth'  # @param {type:'string'}
DRIVE_EXPORT_ROOT_VALUE = '/content/drive/MyDrive/mrixfields_task3_exports/training'  # @param {type:'string'}

RUNTIME_ROOT = Path(RUNTIME_ROOT_VALUE)
DRIVE_CODE_SOURCE = Path(DRIVE_CODE_SOURCE_VALUE)
DRIVE_PROSPECTIVE_SOURCE = Path(DRIVE_PROSPECTIVE_SOURCE_VALUE)
DRIVE_RETROSPECTIVE_SOURCE = Path(DRIVE_RETROSPECTIVE_SOURCE_VALUE)
DRIVE_INITIAL_CHECKPOINT = Path(DRIVE_INITIAL_CHECKPOINT_VALUE)
DRIVE_EXPORT_ROOT = Path(DRIVE_EXPORT_ROOT_VALUE)
CODE_DIR = RUNTIME_ROOT / 'code'
DATA_DIR = RUNTIME_ROOT / 'data'
DATA_VIEW_DIR = RUNTIME_ROOT / 'data_view'
PREPROCESSED_DIR = RUNTIME_ROOT / 'preprocessed'
RESULTS_DIR = RUNTIME_ROOT / 'results'
OUTPUT_DIR = RESULTS_DIR / 'training_runs'
REPORT_DIR = RESULTS_DIR / 'reports'
WEIGHTS_DIR = RUNTIME_ROOT / 'weights'
OFFICIAL_REPO = RUNTIME_ROOT / 'official' / 'MRIxFields2026'
HELPER = CODE_DIR / 'task3_colab_train.py'
REQUIREMENTS = CODE_DIR / 'requirements-task3-training-colab.txt'

drive_root = Path('/content/drive').resolve()
if RUNTIME_ROOT.resolve() == drive_root or drive_root in RUNTIME_ROOT.resolve().parents:
    raise ValueError('RUNTIME_ROOT must be under /content, not mounted Drive.')
for path in (CODE_DIR, DATA_DIR, OUTPUT_DIR, REPORT_DIR, WEIGHTS_DIR):
    path.mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(RUNTIME_ROOT / '.cache' / 'torch')
os.environ['HF_HOME'] = str(RUNTIME_ROOT / '.cache' / 'huggingface')

for filename in ('task3_colab_train.py', 'requirements-task3-training-colab.txt'):
    target = CODE_DIR / filename
    source = next(
        (
            candidate for candidate in
            (DRIVE_CODE_SOURCE / filename, Path('/content') / filename)
            if candidate.is_file()
        ),
        None,
    )
    if source is not None:
        shutil.copy2(source, target)
for required in (HELPER, REQUIREMENTS):
    if not required.is_file():
        raise FileNotFoundError(
            f'Missing {required.name}; put it in {DRIVE_CODE_SOURCE} or /content.'
        )
print('Training runtime:', RUNTIME_ROOT)
print('Persistent export (unused until final cell):', DRIVE_EXPORT_ROOT)


## Install only Task 3 training dependencies

Colab supplies the CUDA PyTorch/torchvision pair. All caches remain inside the
training runtime tree.


In [ ]:
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQUIREMENTS)
])
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before training.')
props = torch.cuda.get_device_properties(0)
print(torch.__version__, torch.version.cuda)
print(props.name, round(props.total_memory / 2**30, 1), 'GiB')


## Stage prospective training data into runtime

This copies only the selected prospective split from Drive into the training
runtime. All later audit, preprocessing, and training commands use DATA_DIR
under /content and never read NIfTI volumes from Drive.


In [ ]:
def require_runtime_path(path):
    path = Path(path).expanduser().resolve()
    drive_root = Path('/content/drive').resolve()
    if path == drive_root or drive_root in path.parents:
        raise ValueError(f'Working path must be runtime-local, not mounted Drive: {path}')
    return path


def stage_nifti_tree(source, destination, expected_files=None):
    source = Path(source).expanduser().resolve()
    destination = require_runtime_path(destination)
    if not source.is_dir():
        raise FileNotFoundError(f'Data source not found: {source}')
    if source == destination or source in destination.parents or destination in source.parents:
        raise ValueError('Source and runtime destination must be separate trees.')
    source_files = sorted(source.rglob('*.nii.gz'))
    if expected_files is not None and len(source_files) != expected_files:
        raise RuntimeError(
            f'Found {len(source_files)} NIfTI files in {source}; expected {expected_files}.'
        )
    if not source_files:
        raise RuntimeError(f'No NIfTI files found in {source}')
    source_index = {
        path.relative_to(source): path.stat().st_size for path in source_files
    }
    destination.mkdir(parents=True, exist_ok=True)
    for index, (relative, size) in enumerate(source_index.items(), start=1):
        target = destination / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.is_file() and target.stat().st_size == size:
            continue
        temporary = target.with_name(f'.{target.name}.part')
        if temporary.exists():
            temporary.unlink()
        shutil.copy2(source / relative, temporary)
        if temporary.stat().st_size != size:
            raise RuntimeError(f'Incomplete runtime copy: {temporary}')
        os.replace(temporary, target)
        if index % 10 == 0 or index == len(source_index):
            print(f'Staged {index}/{len(source_index)} files')
    runtime_index = {
        path.relative_to(destination): path.stat().st_size
        for path in destination.rglob('*.nii.gz')
    }
    if runtime_index != source_index:
        raise RuntimeError('Runtime data inventory differs from the selected source tree.')
    print(f'Runtime data ready: {destination} ({len(runtime_index)} files)')
    return destination


STAGE_PROSPECTIVE_DATA = False  # @param {type:'boolean'}
RUNTIME_PROSPECTIVE_DIR = DATA_DIR / 'training_prospective'
if STAGE_PROSPECTIVE_DATA:
    stage_nifti_tree(
        DRIVE_PROSPECTIVE_SOURCE,
        RUNTIME_PROSPECTIVE_DIR,
        expected_files=45,
    )
elif not RUNTIME_PROSPECTIVE_DIR.is_dir():
    print('Prospective staging is OFF. Keep it off until the baseline milestone.')
else:
    print('Reusing runtime prospective data:', RUNTIME_PROSPECTIVE_DIR)


## Fetch the pinned organizer code

The training helper refuses a mismatched revision/config. This checkout is
runtime-local and separate from pretrained results.


In [ ]:
OFFICIAL_COMMIT = 'f0f4ab2c54ab878b48ddfea8d13f4da512ca411e'
remote_line = subprocess.check_output([
    'git', 'ls-remote', 'https://github.com/MRIxFields/MRIxFields2026.git',
    'refs/heads/main',
], text=True).strip()
live_official_commit = remote_line.split()[0]
if live_official_commit != OFFICIAL_COMMIT:
    raise RuntimeError(
        f'Official main changed to {live_official_commit}; review and update '
        'the training code/config/checkpoint pins together before training.'
    )
if not (OFFICIAL_REPO / '.git').is_dir():
    OFFICIAL_REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([
        'git', 'clone', 'https://github.com/MRIxFields/MRIxFields2026.git',
        str(OFFICIAL_REPO),
    ])
subprocess.check_call([
    'git', '-C', str(OFFICIAL_REPO), 'checkout', '--detach', OFFICIAL_COMMIT
])
revision = subprocess.check_output(
    ['git', '-C', str(OFFICIAL_REPO), 'rev-parse', 'HEAD'], text=True
).strip()
assert revision == OFFICIAL_COMMIT, revision
print('Official training revision:', revision)


## Audit the runtime copy and extract training slices

The audit and canonical-casing compatibility view now point only at runtime
data. Slice extraction is opt-in and also writes only under /content.


In [ ]:
if not RUNTIME_PROSPECTIVE_DIR.is_dir():
    raise RuntimeError('Stage the prospective data into runtime first.')
subprocess.check_call([
    sys.executable, str(HELPER), 'audit-raw',
    '--data-dir', str(DATA_DIR), '--splits', 'pro_train',
    '--report', str(REPORT_DIR / 'pro_train_raw_audit.json'),
])
subprocess.check_call([
    sys.executable, str(HELPER), 'make-data-view',
    '--data-dir', str(DATA_DIR), '--view-dir', str(DATA_VIEW_DIR),
    '--splits', 'pro_train',
])


In [ ]:
RUN_PROSPECTIVE_PREPROCESS = False  # @param {type:'boolean'}
if RUN_PROSPECTIVE_PREPROCESS:
    subprocess.check_call([
        sys.executable, str(HELPER), 'preprocess',
        '--official-repo', str(OFFICIAL_REPO),
        '--data-dir', str(DATA_DIR),
        '--view-dir', str(DATA_VIEW_DIR),
        '--preprocessed-dir', str(PREPROCESSED_DIR),
        '--splits', 'pro_train',
    ])
    subprocess.check_call([
        sys.executable, str(HELPER), 'audit-cache',
        '--preprocessed-dir', str(PREPROCESSED_DIR),
        '--splits', 'pro_train',
        '--report', str(REPORT_DIR / 'pro_train_cache_audit.json'),
    ])
else:
    print('Prospective preprocessing is OFF.')


## Baseline milestone gate and optional runtime initialization

Leave the milestone false until the separate pretrained notebook has produced
and validated the 180-output task3.zip. The official prospective checkpoint
must not initialize an honest held-out prospective pilot because it already saw
those subjects.


In [ ]:
BASELINE_MILESTONE_COMPLETE = False  # @param {type:'boolean'}
STAGE_INITIAL_CHECKPOINT = False  # @param {type:'boolean'}
HF_FILE = 'task3_any_to_any_multimodal/stargan_v2/pro_pretrained/weights/checkpoint_epoch50.pth'
EXPECTED_CHECKPOINT_SIZE = 631346455
EXPECTED_CHECKPOINT_SHA256 = 'c2f8172c1fbdbf0426e22c7d3d2014632f175a38cf818f95e72b1917badabdeb'
INITIAL_CHECKPOINT = WEIGHTS_DIR / HF_FILE
if STAGE_INITIAL_CHECKPOINT:
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Baseline milestone is not acknowledged.')
    if not DRIVE_INITIAL_CHECKPOINT.is_file():
        raise FileNotFoundError(DRIVE_INITIAL_CHECKPOINT)
    INITIAL_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    temporary = INITIAL_CHECKPOINT.with_name('.checkpoint_epoch50.pth.part')
    shutil.copy2(DRIVE_INITIAL_CHECKPOINT, temporary)
    os.replace(temporary, INITIAL_CHECKPOINT)
if INITIAL_CHECKPOINT.is_file():
    digest = hashlib.sha256()
    with INITIAL_CHECKPOINT.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    if INITIAL_CHECKPOINT.stat().st_size != EXPECTED_CHECKPOINT_SIZE:
        raise RuntimeError('Initialization checkpoint size mismatch.')
    if digest.hexdigest() != EXPECTED_CHECKPOINT_SHA256:
        raise RuntimeError('Initialization checkpoint SHA-256 mismatch.')
    print('Runtime initialization ready:', INITIAL_CHECKPOINT)
else:
    print('No initialization staged. The honest pilot can start from scratch.')
print('Training unlocked:', BASELINE_MILESTONE_COMPLETE)


## Pilot training and held-out training results

This leakage-free pilot starts from scratch, withholds one complete prospective
subject, and logs bounded slice-level nRMSE/SSIM/LPIPS proxies by modality.
These are training holdout proxies, not public-validation leaderboard scores.


In [ ]:
RUN_PILOT = False  # @param {type:'boolean'}
PILOT_EPOCHS = 3  # @param {type:'integer'}
PILOT_MICROBATCHES = 100  # @param {type:'integer'}
if RUN_PILOT:
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Baseline milestone is not acknowledged.')
    subprocess.check_call([
        sys.executable, str(HELPER), 'finetune',
        '--official-repo', str(OFFICIAL_REPO),
        '--preprocessed-dir', str(PREPROCESSED_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--run-name', 'pilot_subject_holdout',
        '--from-scratch', '--holdout-subject', 'auto',
        '--epochs', str(PILOT_EPOCHS),
        '--max-steps-per-epoch', str(PILOT_MICROBATCHES),
        '--batch-size', '1', '--gradient-accumulation', '1',
        '--workers', '2', '--save-every-steps', '50',
        '--max-validation-batches', '120',
        '--baseline-milestone-complete',
    ])
else:
    print('Pilot training is OFF.')


In [ ]:
import pandas as pd
pilot_log = (
    OUTPUT_DIR / 'task3_any_to_any_multimodal' / 'stargan_v2'
    / 'pilot_subject_holdout' / 'train.jsonl'
)
rows = []
if pilot_log.is_file():
    for line in pilot_log.read_text().splitlines():
        record = json.loads(line)
        if 'holdout' not in record:
            continue
        groups = {
            'ALL': record['holdout']['overall'],
            **record['holdout']['by_modality'],
        }
        for modality, metrics in groups.items():
            rows.append({
                'epoch': record['epoch'],
                'modality': modality,
                'nRMSE': metrics['slice_proxy_nrmse'],
                'SSIM': metrics['slice_proxy_ssim'],
                'LPIPS': metrics['slice_proxy_lpips'],
                'samples': metrics['samples'],
            })
    display(pd.DataFrame(rows).sort_values(['epoch', 'modality']))
else:
    print('Run the pilot first:', pilot_log)


## Optional all-subject continuation

This is separate from the honest holdout pilot. It continues the released
checkpoint on all prospective subjects and has no honest local prospective
holdout. Start only after the baseline milestone and a measured pilot plan.


In [ ]:
RUN_FULL_FINETUNE = False  # @param {type:'boolean'}
FULL_EPOCHS = 1  # @param {type:'integer'}
MAX_RUNTIME_MINUTES = 300  # @param {type:'integer'}
if RUN_FULL_FINETUNE:
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Baseline milestone is not acknowledged.')
    if not INITIAL_CHECKPOINT.is_file():
        raise FileNotFoundError('Stage the initialization checkpoint first.')
    subprocess.check_call([
        sys.executable, str(HELPER), 'finetune',
        '--official-repo', str(OFFICIAL_REPO),
        '--preprocessed-dir', str(PREPROCESSED_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--run-name', 'pro_pretrained_all_subjects',
        '--init-checkpoint', str(INITIAL_CHECKPOINT),
        '--holdout-subject', 'none',
        '--epochs', str(FULL_EPOCHS), '--batch-size', '1',
        '--gradient-accumulation', '8', '--workers', '2',
        '--save-every-steps', '500', '--keep-last', '3',
        '--max-runtime-minutes', str(MAX_RUNTIME_MINUTES),
        '--baseline-milestone-complete',
    ])
else:
    print('All-subject continuation is OFF.')


## Optional retrospective staging and pretraining

This remains off and gated. It copies the retrospective split into runtime
before any preprocessing; it never trains against Drive-backed files.


In [ ]:
STAGE_RETROSPECTIVE_DATA = False  # @param {type:'boolean'}
RUN_RETROSPECTIVE_PREPROCESS = False  # @param {type:'boolean'}
RUN_RETROSPECTIVE_PRETRAIN = False  # @param {type:'boolean'}
RUNTIME_RETROSPECTIVE_DIR = DATA_DIR / 'training_retrospective'
if STAGE_RETROSPECTIVE_DATA:
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Baseline milestone is not acknowledged.')
    stage_nifti_tree(
        DRIVE_RETROSPECTIVE_SOURCE, RUNTIME_RETROSPECTIVE_DIR
    )
    subprocess.check_call([
        sys.executable, str(HELPER), 'make-data-view',
        '--data-dir', str(DATA_DIR), '--view-dir', str(DATA_VIEW_DIR),
        '--splits', 'retro_train',
    ])
if RUN_RETROSPECTIVE_PREPROCESS:
    if not RUNTIME_RETROSPECTIVE_DIR.is_dir():
        raise RuntimeError('Stage retrospective data into runtime first.')
    subprocess.check_call([
        sys.executable, str(HELPER), 'preprocess',
        '--official-repo', str(OFFICIAL_REPO),
        '--data-dir', str(DATA_DIR), '--view-dir', str(DATA_VIEW_DIR),
        '--preprocessed-dir', str(PREPROCESSED_DIR),
        '--splits', 'retro_train',
    ])
if RUN_RETROSPECTIVE_PRETRAIN:
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Baseline milestone is not acknowledged.')
    subprocess.check_call([
        sys.executable, str(HELPER), 'pretrain',
        '--official-repo', str(OFFICIAL_REPO),
        '--preprocessed-dir', str(PREPROCESSED_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--run-name', 'retro_scratch_colab',
        '--total-steps', '200000', '--batch-size', '1',
        '--workers', '2', '--save-every', '1000', '--keep-last', '3',
        '--max-runtime-minutes', '300',
        '--baseline-milestone-complete',
    ])
if not any((
    STAGE_RETROSPECTIVE_DATA,
    RUN_RETROSPECTIVE_PREPROCESS,
    RUN_RETROSPECTIVE_PRETRAIN,
)):
    print('Retrospective workflow is OFF.')


## Export one selected training result

This is the notebook's only Drive write. Select one runtime run and one
checkpoint. The cell copies that checkpoint plus its provenance and metric log;
it does not export caches, preprocessed slices, raw data, or every checkpoint.


In [ ]:
EXPORT_TRAINING_RESULT = False  # @param {type:'boolean'}
RUN_NAME_TO_EXPORT_VALUE = 'pilot_subject_holdout'  # @param {type:'string'}
CHECKPOINT_TO_EXPORT_VALUE = ''  # @param {type:'string'}
EXPORT_NAME_VALUE = 'pilot_subject_holdout'  # @param {type:'string'}
if EXPORT_TRAINING_RESULT:
    run_dir = (
        OUTPUT_DIR / 'task3_any_to_any_multimodal' / 'stargan_v2'
        / RUN_NAME_TO_EXPORT_VALUE
    )
    checkpoint = Path(CHECKPOINT_TO_EXPORT_VALUE).expanduser()
    if not run_dir.is_dir():
        raise FileNotFoundError(run_dir)
    if not checkpoint.is_file() or RUNTIME_ROOT.resolve() not in checkpoint.resolve().parents:
        raise FileNotFoundError(
            'Choose one checkpoint from the runtime training result tree.'
        )
    export_dir = DRIVE_EXPORT_ROOT / EXPORT_NAME_VALUE
    if export_dir.exists():
        raise FileExistsError(
            f'Export already exists; choose a new name: {export_dir}'
        )
    export_dir.mkdir(parents=True)
    selected = [
        run_dir / 'provenance.json',
        run_dir / 'train.jsonl',
        checkpoint,
    ]
    for source in selected:
        if source.is_file():
            shutil.copy2(source, export_dir / source.name)
    exported_checkpoint = export_dir / checkpoint.name
    def sha256_path(path):
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
                digest.update(chunk)
        return digest.hexdigest()

    source_hash = sha256_path(checkpoint)
    export_hash = sha256_path(exported_checkpoint)
    if source_hash != export_hash:
        raise RuntimeError('Exported checkpoint hash mismatch.')
    (export_dir / 'export.json').write_text(json.dumps({
        'run_name': RUN_NAME_TO_EXPORT_VALUE,
        'checkpoint': checkpoint.name,
        'checkpoint_sha256': source_hash,
    }, indent=2) + '\n')
    print('Saved selected training result:', export_dir)
else:
    print('Training export is OFF; all results remain runtime-local.')
